# TerraScout AI — Producto 1: Exploración y Análisis Geoespacial (EDA)

Notebook estático y 100% autocontenido (compatible directamente con **Google Colab** y **Jupyter local**). Trabaja directamente sobre el dataset oficial depurado (`894` terrenos reales de Calamuchita y Punilla, Córdoba).

## 1. Objetivo, Dataset y Reglas de Negocio

- **Origen de datos:** Publicaciones reales de terrenos en **Mercado Libre (`1.200` → `894` únicos)** cruzadas geográficamente con **campings (`288`)**, demanda en **Google Maps (`237` lugares y `1.754` reseñas)** y topografía satelital (**16 localidades**).
- **Reglas de negocio activas:**
  - Superficie apta: `500` a `20.000 m²`
  - Pendiente máxima recomendada: `<= 12°`
  - Descuento mínimo objetivo: `>= 20%` por debajo de la mediana zonal (`es_oportunidad_p1`)
- **Relación con Producto 2:** Este dataset limpio (`dataset_limpio.csv`) y las correlaciones calculadas aquí contra `log(precio_m2_usd)` fundamentan las variables del modelo hedónico.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_selection import mutual_info_regression

DATASET_URL = 'https://drive.google.com/uc?export=download&confirm=t&id=1y8F8WmbvaXI7QFljrk7KOJk0AagZq0Ji'
LOCAL_CANDIDATES = [
    Path('data/processed/dataset_limpio.csv'),
    Path('../data/processed/dataset_limpio.csv'),
]
local_path = next((p for p in LOCAL_CANDIDATES if p.is_file()), None)
df_clean = pd.read_csv(local_path if local_path is not None else DATASET_URL)

print(f'Fuente cargada: {local_path if local_path is not None else "Google Drive público"}')
print(f'Registros limpios: {len(df_clean)} lotes x {df_clean.shape[1]} columnas')
print(f'Oportunidades P1 (descuento >= 20% vs mediana zonal): {int(df_clean["es_oportunidad_p1"].sum())}')
display(df_clean[['precio_usd', 'superficie_m2', 'precio_m2_usd', 'pendiente_media_grados', 'score_servicios']].describe().round(2))

## 2. Resumen por Localidad y Riesgo Ambiental

Comparación de mediana de precio (`USD/m²`), pendiente media, competencia cercana (`campings_5km`), demanda proxy (`nps_proxy_zona`) y oportunidades detectadas por localidad.

In [ ]:
resumen_loc = (
    df_clean.groupby(['valle_corredor', 'barrio_localidad', 'riesgo_ambiental'], as_index=False)
    .agg(
        lotes=('id_registro', 'count'),
        oportunidades_p1=('es_oportunidad_p1', 'sum'),
        mediana_usd_m2=('precio_m2_usd', 'median'),
        pendiente_media=('pendiente_media_grados', 'mean'),
        campings_5km_prom=('campings_5km', 'mean'),
        nps_proxy=('nps_proxy_zona', 'mean'),
    )
    .sort_values(['oportunidades_p1', 'lotes'], ascending=False)
    .round(2)
)
display(resumen_loc)

## 3. Correlaciones Reales con `log(precio_m2_usd)`

Se evalúan **Pearson**, **Spearman**, **Kendall** e **Información Mutua** sobre los `894` lotes reales para identificar qué variables explican el valor por `m²` antes de pasar al modelado hedónico.

In [ ]:
y_log = np.log(df_clean['precio_m2_usd'].astype(float))
candidate_cols = [
    'superficie_m2', 'lat_eff', 'lon_eff', 'score_servicios',
    'tiene_escritura', 'tiene_arboleda', 'tiene_rio',
    'pendiente_media_grados', 'ndvi_sombra_verano', 'distancia_rio_m',
    'campings_5km', 'distancia_camping_min_km', 'nps_proxy_zona',
]
cols_ok = [c for c in candidate_cols if c in df_clean.columns]
X_corr = df_clean[cols_ok].apply(pd.to_numeric, errors='coerce').fillna(0.0)
mi = mutual_info_regression(X_corr, y_log, random_state=42)

rows_corr = []
for col, mi_val in zip(cols_ok, mi):
    x = X_corr[col]
    rows_corr.append({
        'variable': col,
        'pearson': round(float(stats.pearsonr(x, y_log).statistic), 4),
        'spearman': round(float(stats.spearmanr(x, y_log).statistic), 4),
        'kendall': round(float(stats.kendalltau(x, y_log).statistic), 4),
        'info_mutua': round(float(mi_val), 4),
    })

df_corr = pd.DataFrame(rows_corr).sort_values('info_mutua', ascending=False).reset_index(drop=True)
display(df_corr)

print('\n--- Top 10 Oportunidades P1 (mayor descuento vs mediana zonal) ---')
cols_top = ['id_registro', 'barrio_localidad', 'precio_usd', 'superficie_m2', 'precio_m2_usd', 'descuento_vs_mediana_pct', 'pendiente_media_grados', 'riesgo_ambiental']
cols_top = [c for c in cols_top if c in df_clean.columns]
display(df_clean[df_clean['es_oportunidad_p1'] == True].sort_values('descuento_vs_mediana_pct', ascending=False)[cols_top].head(10))